# Import Libraries

In [32]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor

# Dataset Preparation

In [33]:
# Import dataset
df = pd.read_csv('./Dataset/London Property Listings Dataset.csv')
df

,Price,Property Type,Bedrooms,Bathrooms,Size,Postcode,Area,Price_Category,Area_Avg_Price
0,330000.0,Apartment,1.0,1.0,518.000000,E14,Eastern,Low,1.001684e+06
1,340000.0,Flat,1.0,1.0,887.498269,E14,Eastern,Low,1.001684e+06
2,340000.0,Apartment,1.0,1.0,934.569040,E14,Eastern,Low,1.001684e+06
3,340000.0,Flat,1.0,1.0,887.498269,E14,Eastern,Low,1.001684e+06
4,340000.0,Flat,1.0,1.0,388.000000,SW20,South Western,Low,1.516724e+06
...,...,...,...,...,...,...,...,...,...
29532,795000.0,Flat,3.0,2.0,840.000000,SW20,South Western,Medium,1.516724e+06
29533,795000.0,Flat,2.0,1.0,887.498269,E14,Eastern,Medium,1.001684e+06
29534,795000.0,Flat,2.0,2.0,753.000000,SE1,South Eastern,Medium,6.921048e+05
29535,795000.0,Flat,2.0,2.0,980.000000,SW11,South Western,Medium,1.516724e+06


In [34]:
# Display dataset information
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29537 entries, 0 to 29536
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Price           29537 non-null  float64
 1   Property Type   29537 non-null  object 
 2   Bedrooms        29537 non-null  float64
 3   Bathrooms       29537 non-null  float64
 4   Size            29537 non-null  float64
 5   Postcode        29537 non-null  object 
 6   Area            29537 non-null  object 
 7   Price_Category  29537 non-null  object 
 8   Area_Avg_Price  29537 non-null  float64
dtypes: float64(5), object(4)
memory usage: 2.0+ MB


In [35]:
# Check for NA values
df.isnull().sum()

Price             0
Property Type     0
Bedrooms          0
Bathrooms         0
Size              0
Postcode          0
Area              0
Price_Category    0
Area_Avg_Price    0
dtype: int64

In [36]:
# Drop columns we don't want to train with
df = df.drop(columns=["Price_Category"], errors="ignore")

In [37]:
# Split X and y
X = df.drop("Price", axis=1)
y = df["Price"]

In [38]:
# Identify feature types
numeric_features = ["Bedrooms", "Bathrooms", "Size", "Area_Avg_Price"]
categorical_features = ["Property Type", "Postcode", "Area"]

In [39]:
# Preprocessor
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# Model Development

## Baseline Testing

In [40]:
# Define Models
models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.0001),
    "RandomForest": RandomForestRegressor(
        n_estimators=250, random_state=42, n_jobs=-1
    ),
    "GradientBoosting": GradientBoostingRegressor(
        random_state=42
    ),
    "XGBoost": XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42,
        tree_method="hist",
    ),
}

In [41]:
# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [42]:
# Train & Evaluate All Models
results = []

for name, model in models.items():
    pipe = Pipeline([
        ("preprocess", preprocessor),
        ("model", model)
    ])
    
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    rmse = mean_squared_error(y_test, preds, squared=False)
    r2 = r2_score(y_test, preds)

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

results_df = pd.DataFrame(results).sort_values(by="RMSE")
results_df.reset_index(drop=True, inplace=True)

print("========== BASELINE MODEL PERFORMANCE ==========")
results_df

c:\Users\andre\anaconda3\envs\gpu_env\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:589: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 3494176789217398.5, tolerance: 1693927760361.4258
  model = cd_fast.sparse_enet_coordinate_descent(


========== BASELINE MODEL PERFORMANCE ==========


,Model,MAE,RMSE,R2
0,XGBoost,277465.606283,4.615784e+05,0.714077
1,RandomForest,266621.784389,4.719164e+05,0.701126
2,GradientBoosting,319857.261192,5.143017e+05,0.645028
3,Lasso,378567.809459,1.059798e+06,-0.507315
4,LinearRegression,378737.516007,1.059848e+06,-0.507458
5,Ridge,378444.396566,1.060999e+06,-0.510733


In [43]:
print(results_df)

              Model            MAE          RMSE        R2
0           XGBoost  277465.606283  4.615784e+05  0.714077
1      RandomForest  266621.784389  4.719164e+05  0.701126
2  GradientBoosting  319857.261192  5.143017e+05  0.645028
3             Lasso  378567.809459  1.059798e+06 -0.507315
4  LinearRegression  378737.516007  1.059848e+06 -0.507458
5             Ridge  378444.396566  1.060999e+06 -0.510733


# Model Tuning for XGBoost

## Basic Tuning

In [44]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import joblib

In [45]:
# XGBoost Base Model
xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    tree_method="hist"
)

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", xgb)
])

In [46]:
# Parameter Search Space
param_grid = {
    "model__n_estimators": [300, 400, 500, 700, 900],
    "model__learning_rate": [0.01, 0.02, 0.03, 0.05],
    "model__max_depth": [5, 6, 7, 8, 9, 10],
    "model__min_child_weight": [1, 2, 3, 4, 5],
    "model__subsample": [0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "model__reg_alpha": [0, 0.01, 0.1, 1, 5, 10],
    "model__reg_lambda": [0.5, 1, 2, 3, 5, 10],
}

In [47]:
# Random Search
search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_grid,
    n_iter=80,        # deeper search
    scoring="neg_root_mean_squared_error",
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

In [48]:
# Fit on the existing train/test split
search.fit(X_train, y_train)

print("Best parameters:")
print(search.best_params_)

Fitting 3 folds for each of 80 candidates, totalling 240 fits
Best parameters:
{'model__subsample': 0.8, 'model__reg_lambda': 3, 'model__reg_alpha': 0, 'model__n_estimators': 900, 'model__min_child_weight': 3, 'model__max_depth': 5, 'model__learning_rate': 0.05, 'model__colsample_bytree': 0.9}


In [49]:
# Evaluate best model
best_model = search.best_estimator_
preds = best_model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
rmse = mean_squared_error(y_test, preds, squared=False)
r2 = r2_score(y_test, preds)

print("\n===== Tuned Model Performance =====")
print(f"MAE  : {mae:,.2f}")
print(f"RMSE : {rmse:,.2f}")
print(f"R²   : {r2:.4f}")

# Save for Flask later
joblib.dump(best_model, "model.pkl")
print("\nModel saved as model.pkl")


===== Tuned Model Performance =====
MAE  : 264,485.28
RMSE : 446,723.79
R²   : 0.7322

Model saved as model.pkl


## Feature Engineering on Postcode

In [50]:
from sklearn.model_selection import train_test_split

df_temp = df.copy().drop(columns=["Price_Category"], errors="ignore")

X = df_temp.drop("Price", axis=1)
y = df_temp["Price"]

# Split BEFORE target encoding to avoid leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [51]:
# Combine train X + y only for safe target encoding
train_df = X_train.copy()
train_df["Price"] = y_train

# Target Encoding: Postcode
postcode_mean = train_df.groupby("Postcode")["Price"].mean()
X_train["Postcode_TE"] = X_train["Postcode"].map(postcode_mean)
X_test["Postcode_TE"] = X_test["Postcode"].map(postcode_mean)

# Target Encoding: Postcode Sector (first 2 digits)
X_train["Postcode_Sector"] = X_train["Postcode"].astype(str).str[:2]
X_test["Postcode_Sector"] = X_test["Postcode"].astype(str).str[:2]

sector_mean = train_df.assign(
    Postcode_Sector=train_df["Postcode"].astype(str).str[:2]
).groupby("Postcode_Sector")["Price"].mean()

X_train["Postcode_Sector_TE"] = X_train["Postcode_Sector"].map(sector_mean)
X_test["Postcode_Sector_TE"] = X_test["Postcode_Sector"].map(sector_mean)

# Target Encoding: Area
area_mean = train_df.groupby("Area")["Price"].mean()
X_train["Area_TE"] = X_train["Area"].map(area_mean)
X_test["Area_TE"] = X_test["Area"].map(area_mean)

print("Target encoding complete.")


Target encoding complete.


In [52]:
# Identify feature types after target encoding
numeric_features = [
    "Bedrooms", "Bathrooms", "Size", "Area_Avg_Price",
    "Postcode_TE", "Postcode_Sector_TE", "Area_TE"
]

categorical_features = [
    "Property Type",
    "Area", 
    "Postcode_Sector"   # low-cardinality categorical
]


In [53]:
# Preprocessor
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


In [54]:
# Deep Hyperparameter Tuning with Target Encoded Features
xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    tree_method="hist"
)

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", xgb)
])

param_grid = {
    "model__n_estimators": [300, 500, 700, 900, 1200, 1500],
    "model__learning_rate": [0.005, 0.01, 0.015, 0.02, 0.03, 0.05],
    "model__max_depth": [4, 5, 6, 7, 8, 9, 10, 12],
    "model__min_child_weight": [1, 2, 3, 4, 5, 6],
    "model__subsample": [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    "model__gamma": [0, 1, 3, 5, 10],
    "model__reg_alpha": [0, 0.001, 0.01, 0.1, 1, 5, 10],
    "model__reg_lambda": [0.1, 1, 2, 5, 10, 15, 20],
}

search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_grid,
    n_iter=150,                 # ~10 minutes
    scoring="neg_root_mean_squared_error",
    cv=3,
    verbose=2,                 # more detailed logs
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, y_train)

print("\nBest params:", search.best_params_)

best_model = search.best_estimator_
preds = best_model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
rmse = mean_squared_error(y_test, preds, squared=False)
r2 = r2_score(y_test, preds)

print("\n===== Deep Tuned Target Encoded Model =====")
print(f"MAE  : {mae:,.2f}")
print(f"RMSE : {rmse:,.2f}")
print(f"R²   : {r2:.4f}")

joblib.dump(best_model, "model.pkl")
print("\nSaved tuned model as model.pkl")

Fitting 3 folds for each of 150 candidates, totalling 450 fits

Best params: {'model__subsample': 0.8, 'model__reg_lambda': 20, 'model__reg_alpha': 5, 'model__n_estimators': 500, 'model__min_child_weight': 5, 'model__max_depth': 12, 'model__learning_rate': 0.02, 'model__gamma': 5, 'model__colsample_bytree': 0.7}

===== Deep Tuned Target Encoded Model =====
MAE  : 254,783.24
RMSE : 434,604.25
R²   : 0.7465

Saved tuned model as model.pkl
